# 02 — Athena External Table (Curated Parquet)

This notebook creates an **Athena external table** over the curated Parquet written by Notebook 01.

Why this exists:
- It supports the **Data Engineering** section of the Design Document.
- It gives you SQL access to curated buoy data (counts, sanity queries, etc.).

If you don't need Athena for your workflow, you can skip this notebook.


In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

In [ ]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r CURATED_PREFIX
%store -r BUOY_IDS

print("Bucket:", bucket)
print("Region:", region)
print("CURATED_PREFIX:", CURATED_PREFIX)
print("BUOY_IDS:", BUOY_IDS)

In [ ]:
# Athena staging dir (query results)
s3_staging_dir = f"s3://{bucket}/athena/staging/"
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)
print("Athena staging dir:", s3_staging_dir)

In [ ]:
database_name = "ndbc_data"
table_name = "curated_stdmet"

# Parquet root (PARTITIONED BY (buoy string) expects folders buoy=XXXX/)
s3_parquet_root = f"s3://{bucket}/{CURATED_PREFIX}/"
print("Parquet root:", s3_parquet_root)

In [ ]:
# Create database
with conn.cursor() as cur:
    cur.execute(f"CREATE DATABASE IF NOT EXISTS {database_name}")
print("Created database (if not exists):", database_name)

In [ ]:
# Create external table (partitioned by buoy)
create_table_sql = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    wind_speed_ms double,
    wave_energy double
)
PARTITIONED BY (`buoy` string)
STORED AS PARQUET
LOCATION '{s3_parquet_root}'
"""

print(create_table_sql)

with conn.cursor() as cur:
    cur.execute(create_table_sql)

print("Created table (if not exists):", f"{database_name}.{table_name}")

In [ ]:
# Load partitions (buoy=XXXX folders)
repair_sql = f"MSCK REPAIR TABLE {database_name}.{table_name}"
print(repair_sql)

with conn.cursor() as cur:
    cur.execute(repair_sql)

print("Repaired partitions.")

In [ ]:
# Sanity query: row counts per buoy
q = f"""
SELECT buoy, COUNT(*) AS n
FROM {database_name}.{table_name}
GROUP BY buoy
ORDER BY n DESC
"""
df_counts = pd.read_sql(q, conn)
df_counts

In [ ]:
# Sample data
q = f"""SELECT * FROM {database_name}.{table_name} LIMIT 10"""
pd.read_sql(q, conn)

In [ ]:
%store database_name
%store table_name